# 🔌 GoodWe Assist — Sprint 03
## Refactory conversacional com framework de agentes (LangChain)

**Disciplina:** Prompt and Artificial Intelligence · **Curso:** Ciência da Computação — 1º ano · **Semestre:** 2026.2
**Desafio:** EV Challenge 2026 — GoodWe / FIAP · **Contexto:** ChargeGrid Intelligence

| Nome | RM |
|---|---|
| Ana Beatriz Berbel Marini | 574176 |
| Gustavo Bonamico Piccoli | 569984 |
| Julian Nayde Moncoski | 572603 |
| Marcelo Francisco Josafá Ribeiro Martins | 573905 |
| Maria Eduarda Medeiros Lemos | 574094 |
| Pietro Lorande da Silva | 569125 |

---

Este notebook demonstra o pipeline da Sprint 03 de ponta a ponta. O código de produção vive no
repositório (`src/`, `eval/`, `legacy/`) — aqui apenas o executamos e mostramos as evidências.


## Célula 1 — Ambiente e dependências

In [ ]:
# Colab/Kaggle: clona o repositório. Local: já estamos na raiz do projeto.
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/pietrolorande01-ELFzen/Challenge-_AI-PROMPT.git"

if not Path("src/agent.py").exists():
    if not Path("Challenge-_AI-PROMPT").exists():
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("Challenge-_AI-PROMPT")

sys.path.insert(0, os.getcwd())
print("📁", os.getcwd())

!pip install -q -r requirements.txt

## Célula 2 — Credencial

**Kaggle:** Add-ons → Secrets → `HUGGING_FACE_API_KEY`
**Colab:** ícone 🔑 → `HUGGING_FACE_API_KEY`
**Local:** arquivo `.env` (que está no `.gitignore`)

Nada de chave escrita no notebook — exposição implica penalidade na entrega.

In [ ]:
from src.config import carregar_token, MODELOS

token = carregar_token("HUGGING_FACE_API_KEY")
print("✅ Credencial carregada (", len(token), "caracteres ).")
print("Modelos disponíveis na grade:", list(MODELOS))

## Célula 3 — Base de conhecimento (RAG)

O RAG deixou de ser chamado dentro da função do chatbot e virou um **retriever** do LangChain,
injetável e substituível. Se os PDFs GoodWe não estiverem montados, entra o corpus consolidado
das Sprints 1/2.

In [ ]:
# Kaggle: os.environ["GOODWE_PDF_DIR"] = "/kaggle/input/<seu-dataset>"
# Colab:  os.environ["GOODWE_PDF_DIR"] = "/content"

from src.knowledge import construir_retriever

retriever = construir_retriever()
docs = retriever.invoke("como configurar balanceamento de carga")
for d in docs:
    print("—", d.metadata.get("chunk_id"), "|", d.page_content[:90].replace("\n", " "), "...")

## Célula 4 — O agente

`GoodWeAgent` encapsula: guardrail de entrada → retriever → prompt v3 → memória do framework →
LLM parametrizada → guardrail de saída, com métricas de latência e tokens em cada turno.

In [ ]:
from src.agent import GoodWeAgent

agente = GoodWeAgent(modelo="qwen", retriever=retriever)
print("Modelo :", agente.cfg.repo_id)
print("Params :", f"temperature={agente.cfg.temperature} top_p={agente.cfg.top_p} max_tokens={agente.cfg.max_tokens}")
print("Prompt :", f"v{agente.versao_prompt}")

## Célula 5 — Memória por sessão gerenciada pelo framework (3+ turnos)

O 4º turno pergunta algo que **só pode ser respondido lembrando do 1º turno**. A memória é do
LangChain (`RunnableWithMessageHistory`), não uma lista mantida por nós.

In [ ]:
import time
from eval.eval_set import CONVERSA_MEMORIA
from src.memory import limpar_sessao

SESSAO = "demo_shopping"
limpar_sessao(SESSAO)

for i, pergunta in enumerate(CONVERSA_MEMORIA, 1):
    r = agente.responder(pergunta, session_id=SESSAO)
    print(f"\n{'='*78}\n🧑 TURNO {i}: {pergunta}\n🤖 {r.texto}")
    print(f"   ⏱ {r.latencia_s}s · {r.tokens_total} tokens · {len(agente.memoria(SESSAO))} mensagens na memória")
    time.sleep(3)

In [ ]:
# Dump da memória da sessão — evidência do Bloco A
for m in agente.memoria(SESSAO):
    print(f"[{m['papel']:>9}] {m['conteudo'][:110]}")

In [ ]:
# Controle negativo: sessão nova NÃO enxerga a conversa anterior.
limpar_sessao("outra_sessao")
r = agente.responder(CONVERSA_MEMORIA[-1], session_id="outra_sessao")
print("🤖", r.texto)
print("\n→ Se a resposta acima não souber quantos carregadores existem, o isolamento por sessão está correto.")

## Célula 6 — Guardrails e testes de segurança

Onze casos: prompt injection direta e indireta, exfiltração de prompt e de credencial, troca de
persona, recusas de domínio (jurídico, financeiro, segurança elétrica) e validação de escopo.

In [ ]:
from src.guardrails import validar_entrada
from eval.security_set import CASOS_SEGURANCA

print(f"{'ID':<5}{'FAMÍLIA':<26}{'CAMADA 1':<12}CATEGORIA")
print("-" * 78)
for c in CASOS_SEGURANCA:
    g = validar_entrada(c["entrada"])
    print(f"{c['id']:<5}{c['familia']:<26}{g.veredito.value:<12}{g.categoria}")

In [ ]:
# Execução completa dos casos de segurança, com avaliação documentada de cada resultado
from eval.run_eval import rodar_seguranca
from src.metrics import ColetorMetricas

resumo_seg = rodar_seguranca("qwen", retriever, ColetorMetricas())
resumo_seg

## Célula 7 — Eval set das Sprints 1/2 reexecutado

O **mesmo** conjunto de 8 casos, agora com nota determinística (0 / 0,5 / 1) — é isso que torna a
tabela antes/depois comparável.

In [ ]:
from eval.run_eval import rodar_qualidade_refatorado, rodar_qualidade_legado
from src.metrics import ColetorMetricas

coletor = ColetorMetricas()
resumo_legado = rodar_qualidade_legado("qwen", retriever, coletor)      # Sprints 1/2
resumo_sprint3 = rodar_qualidade_refatorado("qwen", retriever, coletor) # Sprint 03

import pandas as pd
pd.DataFrame({"Sprints 1/2 (legado)": resumo_legado, "Sprint 03 (LangChain)": resumo_sprint3})

## Célula 8 — Comparação entre modelos

Dois modelos pelo mesmo pipeline, mesmo prompt, mesmo retriever, mesmos casos.

In [ ]:
resultados_modelos = {}
for modelo in ["qwen", "llama"]:
    resultados_modelos[modelo] = rodar_qualidade_refatorado(modelo, retriever, ColetorMetricas())

import pandas as pd
pd.DataFrame(resultados_modelos)

In [ ]:
# Varredura de parametrização no modelo escolhido (justifica temperature/top_p/max_tokens)
from eval.run_eval import rodar_parametros

varredura = rodar_parametros("qwen", retriever)
pd.DataFrame([{**v["parametros"], **v["resumo"]} for v in varredura["varreduras"]])

## Célula 9 — Gerar os relatórios com os números medidos

In [ ]:
!python -m eval.run_eval --tudo --modelos qwen llama
!python scripts/gerar_relatorio.py

## Célula 10 — Interface Gradio

Cada aba do navegador recebe um `session_id` próprio: a memória é do framework, não do componente
de interface.

In [ ]:
from src.app import construir_interface

construir_interface(modelo="qwen").launch(share=True)